#  Airbnb Listings — Deep Dive Analysis

In [1]:
import pandas as pd
from scripts.data_loader import load_data, explore_data, show_sample, show_column_summary
from scripts.feature_extraction import extract_all_features
from scripts.data_cleaning import clean_all_columns
from scripts.utils import prepare_analysis_df, save_analysis_df, load_analysis_df, parquet_exists


### Loading Dataset

In [2]:
df = load_data()

[OK] Data loaded successfully: 953 rows, 7 columns


### Initial Data Exploration

In [3]:
explore_data(df)

INITIAL DATA EXPLORATION

>> Shape: 953 rows x 7 columns

>> Data Types & Non-Null Counts:
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Title                   953 non-null    str  
 1   Detail                  953 non-null    str  
 2   Date                    953 non-null    str  
 3   Price(in dollar)        953 non-null    str  
 4   Offer price(in dollar)  166 non-null    str  
 5   Review and rating       947 non-null    str  
 6   Number of bed           953 non-null    str  
dtypes: str(7)
memory usage: 146.8 KB

>> Missing Values:
----------------------------------------
                        Missing  Percentage
Offer price(in dollar)      787       82.58
Review and rating             6        0.63

>> Duplicate Rows: 34 (3.57%)

>> Unique Values Per Column:
------------------------

In [4]:
show_sample(df)


>> First 5 rows:


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed


### Feature Extraction `(7 → 29 columns)`

Apply `extract_all_features()` from `feature_extraction.py`.  
This parses the unstructured `Title` and `Detail` columns using **regex patterns and keyword scoring** to produce 22 new analysis-ready columns:

| Group | New Columns |
|---|---|
| **From Title** | `Property_Type`, `City`, `State`, `Country` |
| **Amenity flags** | `Has_Hot_Tub`, `Has_Sauna`, `Has_Pool`, `Has_BBQ`, `Has_Fireplace`, `Has_Parking`, `Has_Spa` |
| **Location flags** | `Has_Waterfront`, `Is_Near_Beach`, `Is_Mountain`, `Is_Rural`, `Is_Urban` |
| **Classification** | `Property_Subtype`, `Theme` |
| **Text metrics** | `Detail_Word_Count`, `Detail_Char_Count`, `Has_Promo`, `Bedroom_Count` |

In [5]:
df = extract_all_features(df)
print("Shape after feature extraction:", df.shape)
df.head()

Shape after feature extraction: (953, 29)


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed,Property_Type,City,State,...,Is_Near_Beach,Is_Mountain,Is_Rural,Is_Urban,Property_Subtype,Theme,Detail_Word_Count,Detail_Char_Count,Has_Promo,Bedroom_Count
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds,Chalet,Skykomish,Washington,...,False,False,False,False,A-Frame,Nature,5,24,False,NaN
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds,Cabin,Hancock,New York,...,False,False,False,False,A-Frame,Nature,7,47,False,NaN
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds,Cabin,West Farmington,Ohio,...,False,False,True,False,A-Frame,Nature,8,49,False,NaN
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds,Home,Blue Ridge,Georgia,...,False,True,False,False,Other,General,10,50,False,NaN
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed,Treehouse,Grandview,Texas,...,False,False,False,False,Treehouse,Luxury,7,50,False,NaN


### Data Cleaning  `(29 → 42 columns)`

Apply `clean_all_columns()` from `data_cleaning.py`.  
This converts the raw string columns into properly-typed numeric and categorical columns, adding 13 more:

| Source Column | New Columns Derived |
|---|---|
| `Price(in dollar)` | `Price` (float) |
| `Offer price(in dollar)` | `Offer_Price` (float), `Discount_Pct` (%) |
| `Review and rating` | `Rating` (float), `Review_Count` (int), `Is_New_Listing` (bool) |
| `Number of bed` | `Bed_Count` (int), `Bed_Type` (str) |
| `Date` | `Month`, `Start_Day`, `End_Day`, `Duration_Nights`, `Price_Per_Night` |

In [6]:
df = clean_all_columns(df)
print("Shape after cleaning:", df.shape)
df.head()

[OK] Data cleaning complete: 42 columns total
Shape after cleaning: (953, 42)


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed,Property_Type,City,State,...,Rating,Review_Count,Is_New_Listing,Bed_Count,Bed_Type,Month,Start_Day,End_Day,Duration_Nights,Price_Per_Night
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds,Chalet,Skykomish,Washington,...,4.85,531.0,False,4,Standard,June,11.0,16.0,5.0,61.2
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds,Cabin,Hancock,New York,...,4.77,146.0,False,4,Standard,June,6.0,11.0,5.0,97.0
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds,Cabin,West Farmington,Ohio,...,4.91,515.0,False,4,Standard,July,9.0,14.0,5.0,23.8
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds,Home,Blue Ridge,Georgia,...,4.94,88.0,False,5,Standard,June,11.0,16.0,5.0,38.4
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed,Treehouse,Grandview,Texas,...,4.99,222.0,False,1,Queen,June,4.0,9.0,5.0,46.4


### Prepare Analysis DataFrame & Save to Parquet

`prepare_analysis_df()` drops the 10 redundant/low-value columns (the original 7 raw strings + `Bedroom_Count`, `Start_Day`, `End_Day`) and reorders the remaining **32 columns** into logical groups:

`Location → Pricing → Reviews → Beds → Dates → Amenities → Location Flags → Classification → Text Metrics`

The cleaned DataFrame is then saved to disk as a **Parquet file** using `fastparquet`, so this expensive pipeline doesn't need to run again.

> Output: `../data/cleaned_and_processed_data.parquet`

In [7]:
# First time — run full pipeline and save:
df = prepare_analysis_df(df)
save_analysis_df(df)

[OK] Analysis DataFrame ready: 953 rows × 32 columns
[OK] Saved analysis DataFrame → c:\Users\lenov\OneDrive\Desktop\ai-project-collection\Airbnb-Listings-Deep-Dive\data\cleaned_and_processed_data.parquet  (953 rows × 32 cols)


'c:\\Users\\lenov\\OneDrive\\Desktop\\ai-project-collection\\Airbnb-Listings-Deep-Dive\\data\\cleaned_and_processed_data.parquet'